## Load Ontology Mapings into Neo4j, line semantic match (CyberBERT+)

<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/KG Construction/3-Ontologies-Line to Neo4j.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import json, numpy as np
from typing import Dict, Any, List, Tuple
from sentence_transformers import SentenceTransformer, util, models
import pandas as pd
from neo4j import GraphDatabase
import math, os

from dotenv import load_dotenv

EMBED_MODEL   = "ehsanaghaei/SecureBERT_Plus"        # swap to your CyberBERT/CyberBERT+ if available
base = models.Transformer("ehsanaghaei/SecureBERT_Plus", max_seq_length=512)
pool = models.Pooling(
    base.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,   # mean pool token embeddings
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)
norm = models.Normalize()            # L2-normalize for cosine similarity

model = SentenceTransformer(modules=[base, pool, norm])

TOPK_PER_ONTO = 8
CONF_THRESH   = 0.75                          

load_dotenv()  # read NEO4J_URI, NEO4J_USER, NEO4J_PASS from .env

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT_Plus and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def fetch_policy_dataframe(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASS, database=NEO4J_DATABASE) -> pd.DataFrame:
    """
    Aligns to your loaders:
      (:Policy {policy_id, title, source_framework, origin, company_id, company_type})
        -[:HAS_UNIT]->
      (:PolicyUnit {unit_id, number, sort_key_str, heading, text, is_heading, level, ...})
        -[:HAS_LINE]->
      (:Line {line_id, text, line_index_in_unit, source_number_raw, ...})

    Produces the CSV-compatible dataframe.
    """
    cypher = """
    MATCH (p:Policy)-[:HAS_UNIT]->(u:PolicyUnit)
    OPTIONAL MATCH (u)-[:HAS_LINE]->(l:Line)
    WITH p, u, l
    RETURN
      p.policy_id                         AS policy_id,
      p.title                              AS policy_title,
      p.source_framework                   AS source_framework,
      p.origin                             AS policy_origin,
      p.version                            AS policy_version,
      u.unit_id                            AS section_id,
      u.heading                            AS section_title,
      coalesce(u.is_heading, false)        AS is_heading,
      l.line_id                            AS clause_id,
      l.text                               AS clause_text,
      l.sort_order                         AS sort_order,
      u.number                            AS section_number,
      /* Prefer the line's source_number_raw; fallback to the unit's number */
      coalesce(l.source_number_raw, u.number) AS clause_section_number,
      p.company_type                       AS company_type,
      p.company_id                         AS company_id
    ORDER BY
      p.policy_id,
      /* preserve document order using PolicyUnit.sort_key_str then line_index_in_unit */
      u.sort_key_str,
      toInteger(coalesce(l.line_index_in_unit, 2147483647))
    """

    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session(database=database) as session:
            rows = [r.data() for r in session.run(cypher)]
    finally:
        driver.close()

    df = pd.DataFrame(rows, columns=[
        "policy_id","policy_title","source_framework","policy_origin","policy_version",
        "section_id","section_title","is_heading", "section_number",
        "clause_id","clause_text","sort_order","clause_section_number",
        "company_type","company_id"
    ])

    # Light cleanup to mirror your CSV style
    if not df.empty:
        for col in ["policy_title","section_title","clause_text","sort_order"]:
            df[col] = df[col].fillna("").astype(str).str.strip()
        df["is_heading"] = df["is_heading"].fillna(False).astype(bool)
        # keep section numbers as strings (to preserve things like "3.10")
        df["clause_section_number"] = df["clause_section_number"].astype(str)
        df["section_number"] = df["section_number"].astype(str)

    return df




In [5]:
def _num_key(s) -> Tuple[int, ...]:
    """Safe numeric tuple key for strings like '1', '1.2', '3.10'; empty -> ()"""
    if s is None:
        return ()
    parts = str(s).split(".")
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except ValueError:
            # ignore non-numeric fragments
            pass
    return tuple(out)

def build_policy_docs(df: pd.DataFrame) -> List[Dict[str, Any]]:
    work = df.copy()

    # Expected columns from Neo4j-aligned extract
    required = [
        "policy_id","policy_title","source_framework","policy_origin",
        "section_id","section_title","section_number","is_heading",
        "clause_id","clause_text","clause_section_number","policy_version",
        "company_type","company_id"
    ]
    missing = [c for c in required if c not in work.columns]
    if missing:
        raise ValueError(f"Dataframe missing required columns: {missing}")

    # Optional but preferred for stable ordering
    has_sort_col = "sort_column" in work.columns

    # Normalize a bit
    work["clause_text"] = work["clause_text"].fillna("").astype(str).str.strip()
    work["section_title"] = work["section_title"].fillna("").astype(str).str.strip()
    work["policy_title"] = work["policy_title"].fillna("").astype(str).str.strip()
    work["is_heading"] = work["is_heading"].fillna(False).astype(bool)
    work["section_number"] = work["section_number"].astype(str).str.strip()
    work["clause_section_number"] = work["clause_section_number"].astype(str).str.strip()

    # Fallback sort keys if sort_column is absent
    if not has_sort_col:
        work["_clause_sortkey"] = work["clause_section_number"].apply(_num_key)
        order_cols = ["section_number", "section_title", "_clause_sortkey"]
    else:
        # Ensure sort_column is sortable; keep original if mixed types
        order_cols = ["sort_column", "section_number", "clause_section_number"]

    docs: List[Dict[str, Any]] = []

    policy_keys = [
        "policy_id","policy_title","source_framework","policy_version",
        "policy_origin","company_type","company_id"
    ]

    for pvals, pdf in work.sort_values(order_cols, kind="stable").groupby(policy_keys, dropna=False):
        (policy_id, policy_title, source_framework, policy_version, policy_origin, company_type, company_id) = pvals

        sections = []
        # Keep section order stable within policy
        section_group_order_cols = ["section_number", "section_title"]
        if has_sort_col:
            # sort_column also stabilizes section order if lines are absent
            section_group_order_cols = ["section_number", "section_title"]

        for (section_id, section_title, section_number), sdf in (
            pdf.sort_values(order_cols, kind="stable")
               .groupby(["section_id","section_title","section_number"], dropna=False)
        ):
            clauses = []
            for _, r in sdf.iterrows():
                if r["is_heading"]:
                    continue
                text = r["clause_text"]
                if not text:
                    continue

                path = f"{policy_title} > {section_number} {section_title}".strip()
                clause_obj = {
                    "id": r["clause_id"],
                    "path": path,
                    "text": text,
                }
                # carry sort for traceability/debug (optional)
                if has_sort_col:
                    clause_obj["sort"] = r["sort_column"]
                clauses.append(clause_obj)

            if clauses:
                sec_obj = {
                    "id": section_id,
                    "number": section_number,
                    "title": section_title,
                    "clauses": clauses
                }
                # carry sort for sections too (optional)
                if has_sort_col:
                    # pick the min sort among this section’s clauses as its order marker
                    sec_obj["sort"] = min(c.get("sort") for c in clauses if "sort" in c) \
                                      if any("sort" in c for c in clauses) else None
                sections.append(sec_obj)

        policy_doc = {
            "id": policy_id,
            "policy_version": policy_version,
            "title": policy_title,
            "source_framework": source_framework,
            "origin": policy_origin,
            "company_type": company_type,
            "company_id": company_id,
            "sections": sections
        }
        docs.append(policy_doc)

    return docs


In [13]:
import json, numpy as np
from pathlib import Path
from typing import Any, Dict, List, Tuple
import torch
from sentence_transformers import SentenceTransformer, models
from transformers import AutoModel, AutoTokenizer


# ---------------------------
# Loader (safe XPU→CPU fallback)
# ---------------------------


def build_securebert(prefer_xpu: bool = True, max_len: int = 256, truncate_dim: int | None = None):
    # Choose device
    dev = "cpu"
    dtype = None
    if prefer_xpu and hasattr(torch, "xpu") and torch.xpu.is_available():
        dev = "xpu"
        dtype = torch.bfloat16   # Saves VRAM on Intel GPUs

    # 1. Load HF model + tokenizer manually
    tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
    hf_model = AutoModel.from_pretrained(EMBED_MODEL, torch_dtype=dtype)

    # 2. Wrap inside ST Transformer **via the auto-model path**
    base = models.Transformer(EMBED_MODEL, max_seq_length=max_len)
    base.auto_model = hf_model            # override internal model with ours
    base.tokenizer = tokenizer            # override tokenizer

    # 3. Add pooling + normalization
    pool = models.Pooling(
        base.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
        pooling_mode_cls_token=False,
        pooling_mode_max_tokens=False
    )
    norm = models.Normalize()

    # 4. Build SentenceTransformer
    try:
        st_model = SentenceTransformer(modules=[base, pool, norm], device=dev, truncate_dim=truncate_dim)
        return st_model, dev
    except RuntimeError as e:
        # XPU OOM fallback → CPU
        if "OUT_OF_DEVICE_MEMORY" in str(e):
            st_model = SentenceTransformer(modules=[base, pool, norm], device="cpu", truncate_dim=truncate_dim)
            return st_model, "cpu"
        raise

    
def encode_batched(model: SentenceTransformer, texts: List[str], device: str,
                   batch_size: int = 16, normalize: bool = True) -> np.ndarray:
    with torch.no_grad():
        return model.encode(
            texts,
            batch_size=batch_size,
            device=device,
            convert_to_numpy=True,          # returns np.ndarray directly
            normalize_embeddings=normalize, # unit vectors → cosine via dot
            show_progress_bar=False
        )

# Simple singleton cache so we don’t rebuild the model each call
_SECUREBERT_SINGLETON: Tuple[SentenceTransformer, str] | None = None
def get_embedder(prefer_xpu: bool = True, max_len: int = 256, truncate_dim: int | None = None
) -> Tuple[SentenceTransformer, str]:
    global _SECUREBERT_SINGLETON
    if _SECUREBERT_SINGLETON is None:
        _SECUREBERT_SINGLETON = build_securebert(prefer_xpu=prefer_xpu, max_len=max_len, truncate_dim=truncate_dim)
    return _SECUREBERT_SINGLETON

# ---------------------------
# Your existing helpers (unchanged)
# ---------------------------
def load_ontology():
    connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(connection_url, echo=False)
    db_ontology = pd.read_sql('SELECT * FROM ontology;', engine)
    rows = []
    for _, row in db_ontology.iterrows():
        rows.append({
            "id": row["id"],
            "label": row["label"],
            "category": row["category"],
            "counterexamples": (row["counterexamples"]),
            "required": bool(row["required"]),
            "descriptor": f"{row['label']}. {row['description']}"
        })
        
    return rows

def flatten_clauses(policy_doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    out = []
    for s in policy_doc.get("sections", []):
        for cl in s.get("clauses", []):
            text = cl.get("text", "") or ""
            if not text.strip():
                continue
            out.append({
                "clause_id": cl["id"],
                "section": s.get("number"),
                "path": cl.get("path") or f"{policy_doc['title']} > {s.get('number')} {s.get('title')}",
                "text": text.strip()
            })
    return out

# ---------------------------
# Updated matcher (uses cached model + batched encode)
# ---------------------------
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

def match_ontology_to_clauses_semantic(
    policy_doc: Dict[str, Any],
    ontology_rows: List[Dict[str, Any]],
    topk: int = TOPK_PER_ONTO,
    prefer_xpu: bool = True,
    max_len: int = 256,
    truncate_dim: int | None = None,
    batch_size: int = 16,
    thresh: float = CONF_THRESH,
    show_progress: bool = True
) -> List[Dict[str, Any]]:

    clauses = flatten_clauses(policy_doc)
    if not clauses or not ontology_rows:
        return []

    # Build (or reuse) the embedder safely; will fall back to CPU if XPU OOMs.
    model, dev = get_embedder(prefer_xpu=prefer_xpu, max_len=max_len, truncate_dim=truncate_dim)

    clause_texts = [c["text"] for c in clauses]
    ont_texts    = [o["descriptor"] for o in ontology_rows]

    # --- Encoding progress ---
    if show_progress:
        enc_pbar = tqdm(total=2, desc="  Encoding (clauses & ontology)", unit="stage")
    else:
        enc_pbar = None

    # Batched encoding (normalized → cosine via dot product)
    clause_emb = encode_batched(
        model, clause_texts, device=dev, batch_size=batch_size, normalize=True
    )
    if enc_pbar: enc_pbar.update(1)

    ont_emb = encode_batched(
        model, ont_texts, device=dev, batch_size=batch_size, normalize=True
    )
    if enc_pbar:
        enc_pbar.update(1)
        enc_pbar.close()

    # [num_ontologies x num_clauses] cosine scores (since vectors are normalized)
    sims = np.matmul(ont_emb, clause_emb.T)

    results: List[Dict[str, Any]] = []

    # --- Scoring progress ---
    pbar = tqdm(total=len(ontology_rows), desc="  Scoring ontology → clauses", unit="node") if show_progress else None

    for oi, o in enumerate(ontology_rows):
        penalties = np.zeros(len(clauses), dtype=np.float32)
        if o.get("counterexamples"):
            lowers = [t.lower() for t in o["counterexamples"]]
            for ci, ct in enumerate(clause_texts):
                lct = ct.lower()
                if any(cx in lct for cx in lowers):
                    penalties[ci] = 0.10  # tune as needed

        scores = sims[oi] - penalties

        k = min(topk, len(scores)) if len(scores) > 0 else 0
        if k > 0:
            # Fast top-k
            top_idx = np.argpartition(-scores, k-1)[:k]
            keep = [(ci, float(scores[ci])) for ci in top_idx if scores[ci] >= thresh]
            keep.sort(key=lambda x: x[1], reverse=True)

            for ci, sc in keep:
                c = clauses[ci]
                results.append({
                    "ontology_id": o["id"],
                    "ontology_label": o["label"],
                    "category": o["category"],
                    "required": o["required"],
                    "clause_id": c["clause_id"],
                    "section": c["section"],
                    "path": c["path"],
                    "text": c["text"],
                    "score": round(sc, 4),
                    "method": f"semantic({EMBED_MODEL})"
                })
        if pbar: pbar.update(1)

    if pbar: pbar.close()

    results.sort(key=lambda r: (r["ontology_id"], -r["score"]))
    return results


In [7]:
# Helpers (generic)
# -------------------------
def sha256(s: str) -> str:
    import hashlib
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def compact(s: str) -> str:
    return re.sub(r"[ \t]+", " ", s).strip()

def make_id(policy_id: str, number: str) -> str:
    safe = number.replace(" ", "").replace("•", "-")
    return f"{policy_id}:{safe}"

In [8]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

In [9]:
def run_cypher(cypher, params=None):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(cypher, **(params or {}))

In [10]:
INDEX_QUERIES = [
    # Ontology index
    """CREATE INDEX ontology_node_id_idx IF NOT EXISTS
       FOR (o:Ontology) ON (o.node_id)""",
    # Line uniqueness on *line_id* (not id)
    """CREATE CONSTRAINT line_line_id_unique IF NOT EXISTS
       FOR (c:Line) REQUIRE c.line_id IS UNIQUE"""
]

UPSERT_MATCHES = """
UNWIND $rows AS row
    OPTIONAL MATCH (o:Ontology {node_id: row.ontology_id})
    OPTIONAL MATCH (c:Line {line_id: row.clause_id})
    WITH row, o, c
    FOREACH (_ IN CASE WHEN o IS NOT NULL AND c IS NOT NULL THEN [1] ELSE [] END |
      MERGE (o)-[r:INFLUENCED_BY]->(c)
        ON CREATE SET
          r.score      = row.score,
          r.cosine     = row.cosine,
          r.cross      = row.cross,
          r.seed_score = row.seed_score,
          r.penalty    = row.penalty,
          r.method     = row.method,
          r.model      = row.model,
          r.threshold  = row.threshold,
          r.policy_id  = row.policy_id,
          r.run_id     = row.run_id,
          r.createdAt  = datetime(),
          r.updatedAt  = datetime()
        ON MATCH SET
          r.score      = CASE WHEN r.score IS NULL OR row.score > r.score THEN row.score ELSE r.score END,
          r.cosine     = coalesce(row.cosine, r.cosine),
          r.cross      = coalesce(row.cross, r.cross),
          r.seed_score = coalesce(row.seed_score, r.seed_score),
          r.penalty    = coalesce(row.penalty, r.penalty),
          r.method     = coalesce(row.method, r.method),
          r.model      = coalesce(row.model, r.model),
          r.threshold  = coalesce(row.threshold, r.threshold),
          r.policy_id  = coalesce(row.policy_id, r.policy_id),
          r.run_id     = coalesce(row.run_id, r.run_id),
          r.updatedAt  = datetime()
    )
    RETURN
      count(*) AS total,
      count(o) AS have_ontology,
      count(c) AS have_clause,
      sum(CASE WHEN o IS NULL OR c IS NULL THEN 1 ELSE 0 END) AS missing,
      collect(CASE WHEN o IS NULL OR c IS NULL THEN row.clause_id END)[..10] AS sample_missing;
"""


In [11]:
from neo4j import GraphDatabase
from typing import List, Dict, Any

def load_matches_with_check(
    driver,
    rows: List[Dict[str, Any]],
    batch_size: int = 1000,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Inserts (Ontology)-[INFLUENCED_BY]->(Line) when both nodes exist,
    and returns a summary of found/missing pairs.
    """

    query = """
    

    """

    total = have_o = have_c = missing = 0
    sample_missing: List[str] = []

    with driver.session(database=NEO4J_DATABASE) as session:
        for q in INDEX_QUERIES:
            session.run(q)
        for i in range(0, len(rows), batch_size):
            batch = rows[i:i+batch_size]
            res = session.run(UPSERT_MATCHES, rows=batch).data()[0]
            total += res["total"]
            have_o += res["have_ontology"]
            have_c += res["have_clause"]
            missing += res["missing"]
            sample_missing += [x for x in res["sample_missing"] if x]

    summary = {
        "total": total,
        "have_ontology": have_o,
        "have_clause": have_c,
        "missing": missing,
        "sample_missing": sample_missing[:10],
    }

    if verbose:
        if summary["missing"] > 0:
            print(f"⚠️ Warning: {summary['missing']} rows had missing Ontology or Line nodes.")

    return None


In [ ]:
ontology = load_ontology()
df_4j = fetch_policy_dataframe()
policy_docs = build_policy_docs(df_4j)
pbar = tqdm(total=len(policy_docs), desc="Processing Policies", unit="doc")
try:
    for policy_doc in policy_docs:
        file_location=policy_doc.get('file_location')
        policy_id=policy_doc.get('id')
        title=policy_doc.get('title')
        version=policy_doc.get('policy_version')

        policy_matches = match_ontology_to_clauses_semantic(policy_doc, ontology, show_progress=False)
        load_matches_with_check(driver=driver, rows=policy_matches)
        pbar.update(1)
finally:
    pbar.close()

Processing Policies:   0%|          | 0/109 [00:00<?, ?doc/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT_Plus and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT_Plus and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
